# Part I: Language Model Training and Comparison

In [1]:
import re
import random
from collections import Counter, defaultdict
import time
import math
from datasets import load_dataset

c:\Users\jenni\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("stanfordnlp/imdb")
train_lines = dataset["train"]["text"]
test_lines = dataset["test"]["text"]

In [3]:
# Tokenization: define the random variables
# ----------------------------
# Before we can talk about probabilities, we must decide what "tokens" are.
# Here we use a deliberately simple tokenizer:
#   - lowercase
#   - split into words and punctuation tokens
# In real systems, tokenization is more complex (e.g., subword tokenization).

def tokenize(line):
    line = line.strip().lower()
    if not line:
        return []
    # Skip Wikipedia section headers like "== History =="
    if re.fullmatch(r"=+\s*.*\s*=+", line):
        return []
    # Extract words and punctuation as tokens
    return re.findall(r"\w+|[^\w\s]", line)

In [4]:
# Build a token stream with sentence boundary markers
# We add:
#   <s>   = start-of-sentence token
#   </s>  = end-of-sentence token
# This allows the model to learn how sentences start and end.

MAX_LINES = 300  # limit for speed (increase if you want a stronger model)

tokens = []
start_train1 = time.time()
for line in train_lines[:MAX_LINES]:
    toks = tokenize(line)
    if toks:
        tokens.extend(["<s>"] + toks + ["</s>"])

print("Total tokens:", len(tokens))

# Vocabulary = set of all unique tokens we observed
vocab = set(tokens)
print("Vocab size:", len(vocab))
end_train1 = time.time()

Total tokens: 86214
Vocab size: 8199


In [5]:
# Choose n for the n-gram model (Markov assumption)
# n = 3 means a trigram model:
#     P(x_{t+1} | x_1,...,x_t)  ≈  P(x_{t+1} | x_{t-1}, x_t)

n = 3  

In [6]:
# Count n-grams: this is the sufficient statistic for MLE
# We store counts in the most direct way:
#   counts[h][w] = count(h, w)
#   hist_counts[h] = count(h)
# where:
#   h is the (n-1)-gram history (a tuple of tokens)
#   w is the next token
start_train2 = time.time()
counts = defaultdict(Counter)  # history -> Counter(next_word)
hist_counts = Counter()        # history -> total next-token count

# Slide-level mapping:
#   data = token stream
#   model = Markov assumption (history length n-1)
#   estimator = MLE = empirical frequency
for i in range(n - 1, len(tokens)):
    # history is the last (n-1) tokens before position i
    history = tuple(tokens[i - (n - 1): i]) if n > 1 else tuple()
    w = tokens[i]

    counts[history][w] += 1
    hist_counts[history] += 1

print("Number of distinct histories:", len(hist_counts))
end_train2 = time.time()

training_time = end_train1 - start_train1 + end_train2 - start_train2
print("Training time duration:", training_time, "seconds")

Number of distinct histories: 42172
Training time duration: 0.4102344512939453 seconds


In [7]:
# MLE probability: \hat P(w | h) = count(h,w) / count(h)
# This is exactly the MLE for a categorical distribution for each history h.
def mle_prob(history_tokens, w):
    # Keep only the last (n-1) tokens as the history (Markov assumption)
    history = tuple(history_tokens[-(n - 1):]) if n > 1 else tuple()

    ch = hist_counts[history]
    if ch == 0:
        # If this history never appeared in training, MLE gives no guidance.
        # With pure MLE (no smoothing), we return 0.
        return 0.0

    return counts[history][w] / ch

In [8]:
# Compute Perplexity
def perplexity(tokens, n):
    log_p = 0
    N = 0

    for i in range(n-1, len(tokens)):
        history = tuple(tokens[i-(n - 1):i]) if n > 1 else tuple()
        w = tokens[i]
        
        p = mle_prob(history, w)
        if p == 0 :
            p = 1e-10 # smoothing to avoid math domain error
            
        log_p = log_p + math.log(p)
        N = N + 1
    
    return math.exp(-log_p/N)

In [9]:
# For calculate perplexity
test_tokens = []
for line in test_lines[:MAX_LINES]:
    toks = tokenize(line)
    if toks:
        test_tokens.extend(["<s>"] + toks + ["</s>"])

print("Total tokens:", len(test_tokens))

perplexity(test_tokens, n)

Total tokens: 85383


73362160.6401627

In [10]:
#loss
loss = math.log(perplexity(test_tokens,n))
print(loss)

18.110918838043688


In [11]:
# Top-k next-token prediction: show the conditional distribution
# ----------------------------
# A language model does NOT output a single word.
# It outputs a distribution over plausible next tokens.
def topk_next(history_tokens, k=10):
    history = tuple(history_tokens[-(n - 1):]) if n > 1 else tuple()
    total = hist_counts[history]
    if total == 0:
        return []

    # Only rank tokens that were observed after this history
    # (with MLE, unseen continuations have probability 0 anyway)
    dist = [(w, c / total) for w, c in counts[history].items()]
    dist.sort(key=lambda x: x[1], reverse=True)
    return dist[:k]

prompt = "i feel"
prompt_toks = tokenize(prompt)
print("\nPrompt:", prompt)
print("History tokens used:", prompt_toks[-(n - 1):] if n > 1 else [])

print("Top next tokens (MLE):")
for w, p in topk_next(prompt_toks, k=10):
    print(f"  {w:>12s}  {p:.4f}")


Prompt: i feel
History tokens used: ['i', 'feel']
Top next tokens (MLE):
           bad  0.2857
          that  0.2857
     compelled  0.1429
          like  0.1429
          real  0.1429


In [14]:
# Simple text generation by sampling from the learned distribution
# ----------------------------
# Generation demonstrates a classic phenomenon:
#   n-gram models can look locally grammatical
#   but often fail at global coherence (topic/long-range constraints)
def sample_next(history_tokens):
    history = tuple(history_tokens[-(n - 1):]) if n > 1 else tuple()
    counter = counts[history]
    if not counter:
        # If the history was never seen, we stop.
        return "</s>"

    # Sample from the categorical distribution defined by empirical counts.
    words = list(counter.keys())
    weights = list(counter.values())

    return random.choices(words, weights=weights, k=1)[0]

# Try generating from a prompt
gen_prompt = "this movie"
gen_tokens = tokenize(gen_prompt)

# Initialize the model history:
# for bigram, the last token is history; for trigram, last two, etc.
history = (["<s>"] * (n - 1) + gen_tokens) if n > 1 else gen_tokens[:]
generated = gen_tokens[:]

NUM_WORDS = 20
start_inf = time.time()
for _ in range(NUM_WORDS):
    w = sample_next(history)
    if w == "</s>":
        break
    generated.append(w)
    history.append(w)

# Minimal detokenization (just to make output readable)
text = " ".join(generated)
text = re.sub(r"\s+([.,!?;:])", r"\1", text)

end_inf = time.time()
inference_time = end_inf - start_inf

print("\nGenerated text:")
print(text)



Generated text:
this movie has a liking for young, budding love, i would have to ask about the alamo, check out


In [13]:
print("inference time:", inference_time, "seconds")

inference time: 0.0008122920989990234 seconds
